# Setup model, tokenizer, and check text

In [20]:
from transformers import AutoTokenizer
from classes.AttentionHead import AttentionHead
from tqdm import tqdm

# Define the model and the tokenizer from the model
model_ckpt = "bert-base-uncased"

# This triggers downloading of tokenizer files which triggers a Future warning, no concern for now
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Define initial text
text = "time flies like an arrow"

# Pre-Encoder

## Tokenize the text

In [2]:
# Convert the text into tokens. Remove [CLS] and [SEP] tokens by setting add_special_tokens = False
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

# Print the fields of the input
print(f'raw inputs tensor: {inputs}\n')
print(f'input_ids:\t\t{inputs.input_ids}\tTokens mapped to integers as per the tokenizer')
print(f'token_type_ids:\t{inputs.token_type_ids}\tUsed for multi-sentence inputs in models like BERT. 0 = 1st sentence, 1 = 2nd sentence, etc.')
print(f'attention_mask:\t{inputs.attention_mask}\tSeparates actual input = 1 from padding = 0 for attention calculation.')

raw inputs tensor: {'input_ids': tensor([[ 2051, 10029,  2066,  2019,  8612]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

input_ids:		tensor([[ 2051, 10029,  2066,  2019,  8612]])	Tokens mapped to integers as per the tokenizer
token_type_ids:	tensor([[0, 0, 0, 0, 0]])	Used for multi-sentence inputs in models like BERT. 0 = 1st sentence, 1 = 2nd sentence, etc.
attention_mask:	tensor([[1, 1, 1, 1, 1]])	Separates actual input = 1 from padding = 0 for attention calculation.


## Convert tokenized text into embeddings

Make simple embeddings first without positional embeddings but then we create custom class for that and see that below

In [3]:
# Import nn for using the embedding layer and AutoConfig for parameters
from torch import nn
from transformers import AutoConfig

# Get the configuration for the model
config = AutoConfig.from_pretrained(model_ckpt)
print("config stores multiple model parameters like vocab_size and embedding_size.")
print(f'\tconfig.vocab_size: {config.vocab_size}\tconfig.hidden_size: {config.hidden_size}')

# We can create an object of the embedding layer from the config
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
print(f"\ntoken_emb is an object of the embedding layer using above params:\t{token_emb}")

# Convert the input_ids to embeddings
inputs_embeds = token_emb(inputs.input_ids)
print(f'\ninputs_embeds represents tokens in embedding space as: {inputs_embeds.shape}\n\tbatch_size = 1 sentence\t\tseq_length = 5 tokens\t\thidden_size = 768 dimensional vector for each token in embedding dimension')

config stores multiple model parameters like vocab_size and embedding_size.
	config.vocab_size: 30522	config.hidden_size: 768

token_emb is an object of the embedding layer using above params:	Embedding(30522, 768)

inputs_embeds represents tokens in embedding space as: torch.Size([1, 5, 768])
	batch_size = 1 sentence		seq_length = 5 tokens		hidden_size = 768 dimensional vector for each token in embedding dimension


In [4]:
print(f'The maximum number of positions in a model given by config.max_position_embeddings denotes how many positions can there be, in this case {config.max_position_embeddings} we use this to create our embeddings class\n')

# Class to embed input properly with token and position embeddings
class Embeddings(nn.Module):
  def __init__(self, config):
      super().__init__()

      # Create token and position embedding layer usig config
      self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size) # 30522x768
      self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size) # 512x768

      # Create normalization and dropout layers
      self.layer_norm = nn.LayerNorm(config.hidden_size, eps=1e-12)
      self.dropout = nn.Dropout()

  def forward(self, input_ids):

      # Get length of sequence by checking how many columns in input_ids
      seq_length = input_ids.size(1)

      # Create position IDs for input sequence using the retreived seq_length
      position_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)

      # Create token and position embeddings
      token_embeddings = self.token_embeddings(input_ids)
      position_embeddings = self.position_embeddings(position_ids)

      # Combine token and position embeddings
      embeddings = token_embeddings + position_embeddings

      # Apply normalization and dropout
      embeddings = self.layer_norm(embeddings)
      embeddings = self.dropout(embeddings)
      return embeddings

# Check the size of output from this layer
embedding_layer = Embeddings(config)
inputs_embeds = embedding_layer(inputs.input_ids)
print(f'Size of output is {inputs_embeds.size()}:\n{inputs_embeds[0]}')

The maximum number of positions in a model given by config.max_position_embeddings denotes how many positions can there be, in this case 512 we use this to create our embeddings class



NameError: name 'torch' is not defined

# Scaled Dot Product Attention

**Self-Attention Mechanism:** In the self-attention mechanism, each token in the input sequence generates three vectors: Query (Q), Key (K), and Value (V). These vectors are derived from the token's initial embedding through learned linear transformations. The Query vector for each token is compared with the Key vectors of all tokens, including itself, by calculating the dot product. This comparison produces attention scores, which determine the relevance of each token in the context of others. The scores are then scaled to prevent softmax saturation and applied to the Value vectors, creating a weighted sum that represents the token in a context-aware manner. The self-attention score for a token with itself reflects its self-relevance, while scores with other tokens capture the contextual influence of the sequence. In the self-attention mechanism, each token calculates how much attention it should pay to every other token in the sequence, including itself.

In [ ]:
# Imports
import torch
from math import sqrt
import torch.nn.functional as F

# We set the query, key and value tensors as our input embeddings
query = key = value = inputs_embeds

# Print size to observe them
print(f"Q,K,V are initially same size as input_embeds and represent tokens:\t{query.size()}")

# Use any of Q,K,V to get the last dimension showing hidden_dims
dim_last = key.size(-1)

# Transpose the key tensor and observe
key_t = key.transpose(1,2)
print(f"\nKey tensor is transposed to {key_t.size()} because matrix multiplying Query tensor with Key tensor Transposed computes the dot product of each token with every token include itself thereby calculating attention scores at step:\n\tscores = torch.bmm(query, key_t).\nThese scores then need to be scaled by the Embedding Dim of {dim_last} to prevent softmax saturation (Getting only 0 or 1) at step:\n\tscores = scores/sqrt(dim_last)\nTherefore the programming step becomes:\n\tscores = torch.bmm(query, key_t)/sqrt(dim_last)")

# 
scores = torch.bmm(query, key_t)/sqrt(dim_last)

# Use torch.bmm to apply matrix-matrix multiply for each batch resulting in [seq_len, seq_len] matrices scaled by sqrt(dim_last) which means for each token in seq the model has calculated attention scores with every other token including self
print(f"\nResulting scores tensor is of size [batch_size, seq_len, seq_len] = {scores.size()}, showing attention scores for each token in the sequence with every other token including itself\n\t{scores}")


# Calculate weights by using softmax func over last dimension so it will apply to each row of our scores matrix independently
weights = F.softmax(scores, dim=-1)
# Observe the weights matrix and see the sum along every row represents 1 so they represent probabilities
print(f"\nSoftmax is applied across the last dim (columns) to give a probability distribution for each token's relative importance to all other tokens. This is our weights tensor where each row sums to 1\n\t{weights}")

# Calculate attention by applying weights to value tensor
attn_outputs = torch.bmm(weights, value)
print(f"\nFinal Attention is calculated by multiplying this Weights tensor with the Value tensor. This gives us the context-aware representation of each token as a weighted sum of all other tokens, meaning that now each token will focus on other token as per the probability calculated in weights thereby making it context aware. This is a clear contextually-aware representation of each token relative to others in our embedding space and the final size is back to {attn_outputs.shape}\n\t{attn_outputs}")

In [ ]:
# Wrap above steps in a func
def scaled_dot_product_attention(query, key, value):
    
    # Matmul Q, K^T and scale by the last dim (embedding dim) 
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(query.size(-1))
    
    # Apply softmax to get attention weights along cols, prob dist
    weights = F.softmax(scores, dim=-1)
    
    # Calculate context-aware attention by matmul W and V tensors
    return torch.bmm(weights, value)

# Test the function
attn_outputs = scaled_dot_product_attention(inputs_embeds, inputs_embeds, inputs_embeds)
print(f"\nAttention Scores of size [batch_size, seq_len, hidden_size] = {attn_outputs.shape}\n\t{attn_outputs}")

# Single Head Attention

In [ ]:
# New class     
class AttentionHead(nn.Module):
    
    # Define the constructor. Hidden size is 768
    # head_dim multiple of no. of head in bert = 12 so 64 cause 768/12 = 64
    def __init__(self, hidden_size, head_dim):
        super().__init__()
        
        # Set the instance variables
        self.hidden_size = hidden_size # 768
        self.head_dim = head_dim # 64
        
        # Linear transform to Q,K,V tensors of [batch_size x seqlen x head_dim] = 1x5x64 for each head to be concatenated along last dim 
        self.q = nn.Linear(self.hidden_size, self.head_dim)
        self.k = nn.Linear(self.hidden_size, self.head_dim)
        self.v = nn.Linear(self.hidden_size, self.head_dim)
        
    # Define the forward method. inputs are of shape [batch_size x seqlen x hidden_dim] = 1x5x768
    def forward(self, hidden_state):
        
        # Linear transform input hidden_state to get Q,K,V tensors = 1x5x64
        q = self.q(hidden_state)
        k = self.k(hidden_state)
        v = self.v(hidden_state)
        
        # Calculate scaled dot product attention using func and return
        attn_output = scaled_dot_product_attention(q, k, v)
        return attn_output

In [ ]:
## Test the AttentionHead class
# Create an instance of AttentionHead using params from config (768 , 768/12 = 64)
attention_head = AttentionHead(config.hidden_size, config.hidden_size // config.num_attention_heads)

# Pass a batch of input embeddings through the attention head
single_head_output = attention_head(inputs_embeds)

# Print the output along with its shape
print(f"\nSingle AttentionHead output is of size [batch_size, seq_len, head_dim] = {single_head_output.size()}\nThis is for 1 AttentionHead we will make 12 of these heads and concatenate along last dim to get [batch_size x seqlen x hidden_size] = 1x5x768 again but this time with context aware representations of each token from all 12 heads")

# Multi-Head Attention 

In [5]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    """
    Implements multi-headed attention mechanism for a Transformer model.

    Attributes:
        heads (nn.ModuleList): List of AttentionHead modules, one for each attention head.
        output_linear (nn.Linear): Linear layer to combine the outputs from all heads back to the embedding dimension.
    """

    def __init__(self, config):
        """
        Initializes the MultiHeadAttention with the given configuration.

        Args:
            config (transformers.PretrainedConfig): Configuration object containing hyperparameters such as hidden_size and num_attention_heads.
        """
        super().__init__()

        # Set the embedding dimension and number of heads from configuration
        self._embed_dim = config.hidden_size # 768
        self._num_heads = config.num_attention_heads # 12

        # Compute the dimension of each head
        self._head_dim = self._embed_dim // self._num_heads # 768/12 = 64

        # Initialize num_heads = 12 AttentionHead modules of size batch_size x seq_len x head_dim = 1x5x64 each
        self._heads = nn.ModuleList([
            AttentionHead(self._embed_dim, self._head_dim) 
            for _ in range(self._num_heads)
        ]) # 12 AttentionHeads of 1x5x64

        # Linear layer doesn't impact first 2 dims of batch_size and seq_len just multiplies by the last dim
        self._output_linear = nn.Linear(self._embed_dim, self._embed_dim) # 768x768

    def forward(self, hidden_state):
        """
        Performs the forward pass of the multi-headed attention mechanism.

        Args:
            hidden_state (torch.Tensor): The input tensor of shape [batch_size, seq_len, embed_dim].

        Returns:
            torch.Tensor: The output tensor after applying multi-headed attention, of shape [batch_size, seq_len, embed_dim].
        """
        # Process the hidden_state through each attention head and concatenate the results along last head_dim
        x = torch.cat([head(hidden_state) for head in self._heads], dim=-1)

        # Apply the output linear layer that basically just does [batch_size x seq_len x embed_dim] x [embed_dim x embed_dim] = [batch_size x seq_len x embed_dim]
        return self._output_linear(x) # 1x5x768


In [6]:
# Check the implementation of MultiHeadAttention
multi_head_attention = MultiHeadAttention(config)

# Pass a batch of input embeddings through the multi-head attention
multi_head_output = multi_head_attention(inputs_embeds)

# Print the output along with its shape
print(f"\nMulti-Head Attention output is of size [batch_size, seq_len, embed_dim] = {multi_head_output.size()}\nThis now has contextually rich hidden states\n{multi_head_output}")

NameError: name 'AttentionHead' is not defined

# Feed-Forward Network

The `FeedForward` network plays a crucial role in transforming and enriching the token embeddings produced by the multi-head attention mechanism. The first linear layer expands each token's embedding from its original dimensionality `hidden_size` = 768 to a larger `intermediate_size` = 3072. This expansion allows the model to capture more complex patterns and interactions that are not possible within the smaller space. The GELU activation function is then applied, introducing non-linearity, which enables the model to learn intricate, non-linear relationships in the data. After this, the second linear layer projects the embedding back to the original `hidden_size`, ensuring that the enriched representation is compatible with the rest of the model's architecture. Finally, a dropout layer is applied to prevent overfitting by randomly zeroing out elements of the tensor during training. Together, these layers allow the model to learn and retain complex features while also maintaining robustness and generalization.


In [7]:
# Class for feedforward layer
class FeedForward(nn.Module):
  def __init__(self, config):
      super().__init__()

      # Set the instance variables
      self._embed_dim = config.hidden_size # hidden_size = 768
      self._expanded_dim = config.intermediate_size # intermediate_size = 3072
      self._drop_prob = config.hidden_dropout_prob # hidden_dropout_prob = 0.1
      
      # Linear layer to expand from hidden_size to intermediate_size
      self._linear_1 = nn.Linear(self._embed_dim, self._expanded_dim) # hidden_size x intermediate_size = 768x3072

      # GELU activation function to introduce non-linearity and capture complex patterns in the data
      self.gelu = nn.GELU()

      # Linear layer to project back to hidden_size
      self.linear_2 = nn.Linear(self._expanded_dim, self._embed_dim) # intermediate_size x hidden_size = 3072x768
      
      # Dropout layer for regularization with a dropout probability of 0.1 meaning 10% of input elements are set to zero
      self.dropout = nn.Dropout(self._drop_prob)

  def forward(self, x):
      
      # First linear transformation to input tensor of shape [batch_size x seq_len x embed_dim] = 1x5x768
      lin1 = self._linear_1(x) # batch_size x seq_len x intermediate_size = 1x5x3072

      # GELU activation function
      nonlin = self.gelu(lin1) # batch_size x seq_len x intermediate_size = 1x5x3072

      # Second linear transformation
      lin2 = self.linear_2(nonlin) # batch_size x seq_len x intermediate_size = 1x5x768

      # Use dropout for regularization 
      output = self.dropout(lin2) # batch_size x seq_len x hidden_size = 1x5x768

      # Return the output after dropout layer, shape remains the same as the input
      return output

# Create an instance of FeedForward using params from config (768, 3072)
feed_forward = FeedForward(config)

# Pass a batch of attention outputs through the feedforward layer
ff_outputs = feed_forward(attn_outputs)

# Print the output along with its shape
print(f"\nFFN output of shape {ff_outputs.size()}\n{ff_outputs}")


NameError: name 'attn_outputs' is not defined

# Full Encoder Layer

Now this will have the full Transformer Encoder Layer which includes the multi-head attention followed by skip connection and norm followed by feed-forward layer and skip connection again.

In [9]:
# Class to represent encoder of transformer with pre norm and add
class EncoderLayer(nn.Module):

    def __init__(self, config):
        super().__init__()
        
        # Set the hidden size
        self._embed_dim = config.hidden_size # hidden_size = 768

        # Setup first normalization block which is before multihead_attn, normalization across last dimension embed_dim = 768
        self._norm1 = nn.LayerNorm(self._embed_dim) # batch_size x seq_len x embed_dim = 100x256x768
        self._multihead_attn = MultiHeadAttention(config) # batch_size x seq_len x embed_dim = 100x256x768
        
        # Setup second normalization block before feed-forward.
        self._norm2 = nn.LayerNorm(self._embed_dim) # batch_size x seq_len x embed_dim = 100x256x768
        self._feed_forward = FeedForward(config) # batch_size x seq_len x embed_dim = 100x256x768
        

    def forward(self, x):
        
        # Apply 1st layer normalization
        hidden_state = self._norm1(x)

        # Get the multihead attention on hidden state
        multihead_attn = self._multihead_attn(hidden_state)

        # Apply skip connection by adding the input x and the output of multihead attention and creating new x2
        x2 = x + multihead_attn

        # Apply 2nd layer normalization before ffn
        hidden_state2 = self._norm2(x2)

        # Get the FFN output on this new hidden state
        ffn_output = self._feed_forward(hidden_state2)

        # Apply skip connection by adding the input of ffn x2 and the output of ffn to get x3
        x3 = x2 + ffn_output

        # Return the final x3
        return x3


In [13]:
# Create an object of EncoderLayer using params from config (768)
encoder_layer = EncoderLayer(config)

# Pass input embeddings through the encoder layer
encoder_layer_output = encoder_layer(inputs_embeds)

# Observe
print(f"\nEncoderLayer output of shape {encoder_layer_output.size()}\n{encoder_layer_output}")


EncoderLayer output of shape torch.Size([1, 5, 768])
tensor([[[ 0.5851,  0.9545,  0.0460,  ..., -0.8469, -1.6975, -0.3560],
         [-1.1227, -0.7831,  3.1586,  ..., -0.8149, -0.2877, -0.4682],
         [-1.2159, -1.6182, -2.0869,  ..., -1.6532,  0.9494,  0.1418],
         [ 0.1312,  1.1008, -1.6963,  ..., -1.5285, -0.5228,  0.2911],
         [ 0.5940,  0.8973,  0.6307,  ...,  0.6951,  2.7458,  0.3659]]],
       grad_fn=<AddBackward0>)


In [21]:
# COMBINE PREV CLASSES FOR ENCODER
class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()

        # Setup the embeddings which are token + positional normalized
        self.embeddings = Embeddings(config)

        # Setup the layers which have the full encoder block from EncoderLayer object
        self.layers = nn.ModuleList([EncoderLayer(config) for _ in range(config.num_hidden_layers)])

    # Define the forward function
    def forward(self, x):

        # Convert input x to embeddings
        x = self.embeddings(x)

        # Loop through each layer
        for layer in tqdm(self.layers,desc="Encoder"):

            # Pass x through the layers setting it to the output
            x = layer(x)

        # Return final x
        return x

# Test the encoder
encoder = Encoder(config)
encoder_output = encoder(inputs.input_ids)

# Show the output shape and first few elements of the output.
print(f"\nEncoder output of shape {encoder_output.size()}\n{encoder_output}")

Encoder:   0%|          | 0/12 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Encoder: 100%|██████████| 12/12 [00:00<00:00, 165.29it/s]


Encoder output of shape torch.Size([1, 5, 768])
tensor([[[-7.6479e-02, -3.3774e-01,  2.2239e+00,  ...,  1.2336e+00,
           1.1864e+00,  7.2846e-01],
         [ 3.0804e-01,  3.8594e+00, -1.0351e+00,  ..., -1.5013e+00,
           1.0168e+00,  1.7581e-03],
         [-2.8078e+00, -9.9750e-02,  1.4305e+00,  ...,  6.1474e-01,
          -1.7734e-01,  6.0309e-01],
         [ 3.3162e+00,  2.6296e-01, -2.1538e-01,  ...,  1.8541e+00,
           2.0720e-01, -3.4829e-01],
         [-6.6617e-01,  4.9326e-01,  1.1938e+00,  ...,  9.4356e-01,
           1.0835e+00, -2.8007e-01]]], grad_fn=<AddBackward0>)


In [18]:
config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.37.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}